In [1]:
!pip install -q langchain langchain-groq gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.8 MB/s eta 0:00:00


In [ ]:
import os
import gradio as gr
from google.colab import userdata
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

# 1. Securely load your Groq API key from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")


def generate_content_variations(user_title):
  """Takes a user-provided title and generates three unique content variations

  using low, medium, and high temperature settings.
  """
  if not user_title.strip():
    return (
        "Please provide a valid title first.",
        "Please provide a valid title first.",
        "Please provide a valid title first.",
    )

  # Define a simple LangChain prompt template
  prompt_template = ChatPromptTemplate.from_messages([
      (
          "system",
          (
              "You are an expert content creator. Write an engaging, well-crafted"
              " short paragraph/content piece based on the user-provided title."
          ),
      ),
      ("human", "Title/Topic: {title}"),
  ])

  output_parser = StrOutputParser()

  # 2. Initialize ChatGroq instances with 3 distinct temperature variations
  llm_low = ChatGroq(model="openai/gpt-oss-120b", temperature=0.1)
  llm_med = ChatGroq(model="openai/gpt-oss-120b", temperature=0.5)
  llm_high = ChatGroq(model="openai/gpt-oss-120b", temperature=0.9)

  # 3. Create simple LangChain LCEL chains
  chain_low = prompt_template | llm_low | output_parser
  chain_med = prompt_template | llm_med | output_parser
  chain_high = prompt_template | llm_high | output_parser

  # 4. Invoke chains with the user's title
  output_low = chain_low.invoke({"title": user_title})
  output_med = chain_med.invoke({"title": user_title})
  output_high = chain_high.invoke({"title": user_title})

  return output_low.strip(), output_med.strip(), output_high.strip()


# 5. Build custom Gradio UI using gr.Blocks for horizontal layout
with gr.Blocks() as demo:
  gr.Markdown("# Multi-Temperature Content Generator")
  gr.Markdown(
      "Powered by **LangChain**, **Groq**, and **Gradio**. Enter a title"
      " below to compare generation outputs side-by-side across three"
      " temperatures."
  )

  # Input Section (Stacked vertically)
  with gr.Column():
    user_input = gr.Textbox(
        lines=2,
        placeholder=(
            "Enter a title or topic (e.g., The Future of Quantum"
            " Computing)..."
        ),
        label="User-Given Title / Topic",
    )
    submit_btn = gr.Button("Generate Variations", variant="primary")

  # Output Section (Horizontally aligned using gr.Row)
  with gr.Row():
    out_low = gr.Textbox(
        label="Low Temp (0.1) — Precise & Factual", lines=7
    )
    out_med = gr.Textbox(
        label="Medium Temp (0.5) — Balanced & Professional", lines=7
    )
    out_high = gr.Textbox(label="High Temp (0.9) — Creative & Bold", lines=7)

  # Bind the button action to the function and components
  submit_btn.click(
      fn=generate_content_variations,
      inputs=user_input,
      outputs=[out_low, out_med, out_high],
  )

# Launch the Gradio app inside Colab with a shareable link
demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1e06740bcceb9fda96.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
